# 05Gb — Whole-model evaluation (Phase G2b, exploratory)

Evaluates the whole-model explanations from **04Ge** along the two axes the per-feature
track (05G) cannot address. Everything reuses the G3 scorers: the split records
(`results/global_whole_split/`) are byte-compatible with the G2a per-feature records, so
the same deterministic rubric (`utils.rubric`) and reference-based judge
(`utils.eval.run_global_judge`, `src_subdir="global_whole_split"`) apply unchanged.

- **Coverage** — how many of the 9 features each whole-model answer actually described
  (the meeting's "only 5 of 9 right" concern, made measurable).
- **Axis 1 — representation** — all 9 curves/plots vs the single beeswarm, reported
  **per GT field** (no aggregate winner); the beeswarm is scored only on the fields it
  can carry (direction, rank) via `fair_total`.
- **Axis 2 — mechanism** — full-push (`json_all`/`vision_all`) vs pull (`tooluse_all`) at
  constant full information.
- **Beeswarm readability** — `vision_beeswarm` vs the info-matched `json_beeswarm`:
  same information, different modality → the pure "can the LLM *read* the swarm?" effect.

> Runs on the **real** split records if 04Ge was run with `RUN_API=True`; otherwise it
> falls back to a **STUB** (deterministic, correct ranks/direction) so every table and
> code path is verified without API cost. The reference judge is separately guarded
> (`RUN_JUDGE`).

In [1]:
from __future__ import annotations

import sys, json, tempfile
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import (
    RESULTS_DIR, EXPLANATIONS_DIR, WHOLE_SPLIT_SUBDIR, WHOLE_CONDITIONS,
    list_global_features, feature_importance_map, describe_curve,
    build_whole_record, write_split_records,
)
from utils import global_eval

XAI_MODELS = ['xgb', 'ebm']
FEATURES   = list_global_features('ebm', explanations_dir=EXPLANATIONS_DIR)

df = global_eval.load_whole_rubric()      # real split records, or None if 04Ge not run
STUB = df is None

if STUB:
    def _stub_expl(model_name):
        imp = feature_importance_map(model_name, explanations_dir=EXPLANATIONS_DIR)
        lines = ['<analysis>stub</analysis>']
        for f in FEATURES:
            d = describe_curve(model_name, f, explanations_dir=EXPLANATIONS_DIR)
            lines += [f'[FEATURE: {f}]',
                      f'[EFFECT] The effect is {d["direction"]} and {d["monotonicity"]}.',
                      f'[IMPORTANCE] Rank {imp[f]["rank"]} of {len(FEATURES)}.']
        lines.append('[RECOMMENDATION] Plan bikes and staff around the top drivers.')
        return '\n'.join(lines)

    recs = []
    for m in XAI_MODELS:
        for c in WHOLE_CONDITIONS:
            txt = _stub_expl(m)
            if c.name == 'vision_beeswarm':               # drop last feature -> coverage < 9
                txt = txt.rsplit(f'[FEATURE: {FEATURES[-1]}]', 1)[0].rstrip() + \
                      '\n[RECOMMENDATION] Plan around the top drivers.'
            recs.append(build_whole_record(condition=c, model_name=m, explanation=txt,
                                           usage={}, llm_model='stub'))
    tmp = Path(tempfile.mkdtemp())
    write_split_records(recs, features=FEATURES, split_dir=tmp / WHOLE_SPLIT_SUBDIR)
    df = global_eval.load_whole_rubric(results_dir=tmp)
    print(f'No real split records found -> STUB verification on {len(df)} rows.')
    print('(Values are placeholders; only the labels/shapes are meaningful until 04Ge runs.)')
else:
    print(f'Loaded {len(df)} real whole-model split records.')

df[['condition', 'xai_model', 'feature', 'form_type', 'direction', 'rank',
    'structure', 'total', 'fair_total', 'dropped']].head(6)

No real split records found -> STUB verification on 90 rows.
(Values are placeholders; only the labels/shapes are meaningful until 04Ge runs.)


,condition,xai_model,feature,form_type,direction,rank,structure,total,fair_total,dropped
0,json_all,ebm,holiday,near-flat,1.0,1.0,0.0,0.6667,0.6667,False
1,json_all,ebm,hr,categorical,0.5,1.0,0.0,0.5000,0.5000,False
2,json_all,ebm,hum,non-monotonic,0.0,1.0,0.0,0.3333,0.3333,False
3,json_all,ebm,mnth,categorical,0.5,1.0,0.0,0.5000,0.5000,False
4,json_all,ebm,temp,non-monotonic,0.0,1.0,0.0,0.3333,0.3333,False
5,json_all,ebm,weathersit,categorical,0.5,1.0,0.0,0.5000,0.5000,False


## 1. Coverage — how many of the 9 features were described?

A whole-model answer that silently drops features is the core scoring risk from the
meeting. `dropped=True` feature blocks are scored as total misses.

In [2]:
cov = global_eval.whole_coverage(df)
print('Features described (of 9), by condition x model:')
print(cov)

Features described (of 9), by condition x model:
xai_model        ebm  xgb
condition                
json_all           9    9
vision_all         9    9
tooluse_all        9    9
json_beeswarm      9    9
vision_beeswarm    8    8


## 2. Axis 1 — representation (all vs beeswarm), per GT field

The two representations carry **different** information, so there is no single winner:
report each GT field. Expectation a priori — all-plots/curves win on `structure`
(shape/peak), the beeswarm stays competitive on `direction`/`rank`. Read the beeswarm
rows on `direction`/`rank` only; `fair_total` already restricts the beeswarm to those.

In [3]:
a1 = global_eval.axis1_representation(df)
print('Mean rubric sub-scores per condition (beeswarm: read direction/rank; fair_total is field-fair):')
print(a1)

Mean rubric sub-scores per condition (beeswarm: read direction/rank; fair_total is field-fair):
                 direction   rank  structure  fair_total
condition                                               
json_all             0.528  1.000      0.111       0.546
vision_all           0.528  1.000      0.111       0.546
tooluse_all          0.528  1.000      0.111       0.546
json_beeswarm        0.528  1.000      0.111       0.764
vision_beeswarm      0.417  0.889      0.111       0.653


## 3. Axis 2 — mechanism (push vs pull) at full information

Restricted to the `all` conditions (constant full information): full-push
(`vision_all`/`json_all`, mechanism = push) vs pull (`tooluse_all`).

In [4]:
a2 = global_eval.axis2_mechanism(df, value='total')
print('Mean rubric total by mechanism x model (all-information conditions only):')
print(a2)

Mean rubric total by mechanism x model (all-information conditions only):
xai_model    ebm    xgb
mechanism              
pull       0.556  0.537
push       0.556  0.537


## 4. Beeswarm readability — image vs info-matched numbers

`vision_beeswarm` (the swarm image) vs `json_beeswarm` (the same ranking + colour
direction + spread as numbers). Same information, so the gap is the **pure modality
effect**: can the LLM read the swarm as well as it reads the equivalent numbers?

In [5]:
bee = global_eval.beeswarm_readability(df, value='fair_total')
print('Mean fair_total (direction+rank) — swarm image vs info-matched numbers:')
print(bee)

Mean fair_total (direction+rank) — swarm image vs info-matched numbers:
xai_model          ebm    xgb
condition                    
json_beeswarm    0.778  0.750
vision_beeswarm  0.667  0.639


## 5. Reference-based judge (billed — guarded)

Scores the whole-model split records with the same reference judge as 05G, via
`src_subdir="global_whole_split"` (no change to the frozen G2a judge outputs). Set
`RUN_JUDGE=True` after 04Ge has produced real split records.

In [6]:
RUN_JUDGE = False   # <- set True to score the whole-model split records (billed)

if RUN_JUDGE and not STUB:
    from utils.eval import run_global_judge
    from utils.llm import ask_text
    JUDGE_MODEL = 'claude-opus-4-8'
    judge_df = run_global_judge(ask_text, JUDGE_MODEL,
                                src_subdir='global_whole_split',
                                out_subdir='global_whole_judge')
    print(f'Judged {len(judge_df)} whole-model split records.')
    display(judge_df.groupby('form_pipeline')[['faithfulness', 'clarity', 'completeness']].mean().round(3))
elif STUB:
    print('STUB mode: skipping the billed judge (run 04Ge with RUN_API=True first).')
else:
    print('RUN_JUDGE=False -> skipped. Set True to score the whole-model split records.')

STUB mode: skipping the billed judge (run 04Ge with RUN_API=True first).
